In [1]:
# """
# Vesuvius competition metric.

# Expects standard Kaggle paths and Linux in order to manage dependencies.
# """

# import glob
# import importlib
# import os
# import subprocess
# import sys
# import numpy as np
# import pandas as pd
# from PIL import Image, ImageSequence


# class ParticipantVisibleError(Exception):
#     pass


# class HostVisibleError(Exception):
#     pass


# def load_volume(path):
#     im = Image.open(path)
#     slices = []
#     for i, page in enumerate(ImageSequence.Iterator(im)):
#         slice_array = np.array(page)
#         slices.append(slice_array)
#     volume = np.stack(slices, axis=0)
#     return volume


# def install_dependencies():
#     """On Kaggle, the topometrics library must be installed during the run. This function handles the entire process."""
#     try:
#         import topometrics.leaderboard

#         return None
#     # The broad exception is necessary as the initial import can fail for multiple reasons.
#     except:
#         pass

#     resources_dir = '/kaggle/input/vesuvius-metric-resources'
#     install_dir = '/kaggle/working/topological-metrics-kaggle'

#     try:
#         subprocess.run(
#             f'cd {resources_dir} && uv pip install --no-index --find-links=wheels -r topological-metrics-kaggle/requirements.txt',
#             shell=True,
#             check=True,
#         )
#         subprocess.run(f'cd /kaggle/working && cp -r {resources_dir}/topological-metrics-kaggle .', shell=True, check=True)
#         subprocess.run(
#             f'cd {install_dir} && chmod +x scripts/setup_submodules.sh scripts/build_betti.sh && make build-betti',
#             shell=True,
#             check=True,
#         )
#         subprocess.run(
#             f'cd {install_dir} && uv pip install -e . --no-deps --no-index --no-build-isolation -v',
#             shell=True,
#             check=True,
#         )
#         # Add the new library to Python's path and invalidate caches to ensure it's found.
#         sys.path.append('/kaggle/working/topological-metrics-kaggle/src')
#         importlib.invalidate_caches()

#     except Exception as err:
#         raise HostVisibleError(f'Failed to install topometrics library: {err}')


# def generate_standard_submission(submission_dir: str) -> None:
#     # Dependencies installed here as generate_standard_submission is the first metric function that gets called by the orchestrator.
#     submission_tifs = glob.glob(f'{submission_dir}/**/*.tif', recursive=True)
#     if len(submission_tifs) == 0:
#         submission_tifs = glob.glob('/kaggle/tmp/**/*.tif', recursive=True)
#     if len(submission_tifs) == 0:
#         raise ParticipantVisibleError('No submission files found')
#     df = pd.DataFrame({'tif_paths': submission_tifs})
#     df['id'] = df['tif_paths'].apply(lambda x: x.split('/')[-1].split('.')[0])
#     os.chdir('/kaggle/working')
#     df[['id', 'tif_paths']].to_csv('submission.csv', index=False)


# def score_single_tif(
#     gt,
#     pr,
#     surface_tolerance = 2,
#     voi_connectivity=26,
#     voi_transform='one_over_one_plus',
#     voi_alpha=0.3,
#     topo_weight=0.3,
#     surface_dice_weight=0.35,
#     voi_weight=0.35,
# ):

#     install_dependencies()
#     # The import is here to ensure dependencies are loaded first.
#     try:
#         # Use a standard import now that the path is reliably set.
#         import topometrics.leaderboard
#     except Exception as err:
#         raise HostVisibleError(f'Failed to import topometrics after installation: {err}')

#     score_report = topometrics.leaderboard.compute_leaderboard_score(
#         predictions=pr,
#         labels=gt,
#         dims=(0, 1, 2),
#         spacing=(1.0, 1.0, 1.0),  # (z, y, x)
#         surface_tolerance=surface_tolerance,  # in spacing units
#         voi_connectivity=voi_connectivity,
#         voi_transform=voi_transform,
#         voi_alpha=voi_alpha,
#         combine_weights=(topo_weight, surface_dice_weight, voi_weight),  # (Topo, SurfaceDice, VOI)
#         fg_threshold=None,  # None => legacy "!= 0"; else uses "x > threshold"
#         ignore_label=2,  # voxels with this GT label are ignored
#         ignore_mask=None,  # or pass an explicit boolean mask
#     )
#     return np.clip(score_report.score, a_min=0.0, a_max=1.0)


In [2]:
import numpy as np
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects


def load_volume(path):
    vol = tifffile.imread(path)          # (D, H, W)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]           # (1, D, H, W, 1)
    return vol


# ==========================================
# ROTATION TTA HELPERS (CLOCKWISE)
# ==========================================
def rot90_volume(vol, k):
    """
    Rotate volume k times 90° clockwise in HW plane.
    vol:
      (1, D, H, W, 1) OR (D, H, W)
    """
    if vol.ndim == 5:
        return np.rot90(vol, k=-k, axes=(2, 3))
    else:
        return np.rot90(vol, k=-k, axes=(1, 2))


def unrot90_volume(vol, k):
    return rot90_volume(vol, (4 - k) % 4)


def predict_probs_tta_rot(sample,m):
    """
    4x rotation TTA: 0°, 90°, 180°, 270°
    sample: (1, D, H, W, 1)
    returns: averaged probs (D, H, W)
    """
    probs_accum = []

    for k in range(4):
        s_rot = rot90_volume(sample, k)
        swi = pred(m)
        out = swi(s_rot)              # (1, D, H, W, 2)
        out = np.asarray(out)
        probs = out[0, ..., 1]         # (D, H, W)

        probs = unrot90_volume(probs, k)
        probs_accum.append(probs)

    return np.mean(probs_accum, axis=0)


# ==========================================
# HELPER: Anisotropic Structure Builder
# ==========================================
def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius

    if z == 0 and r == 0:
        return None

    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct

    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct

    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


# ==========================================
# MAIN POST-PROCESSING LOGIC
# ==========================================
def topo_postprocess(
    probs,          # (D, H, W)
    T_low=0.50,
    T_high=0.90,
    z_radius=0,
    xy_radius=3,
    dust_min_size=100,
):
    # --- Step 1: 3D Hysteresis ---
    strong = probs >= T_high
    weak   = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # --- Step 2: 3D Anisotropic Closing ---
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # --- Step 3: Dust Removal ---
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)


# ==========================================
# PREDICT (WITH ROTATION TTA)
# ==========================================
def predict(
    sample,
    iid=None,
    T_low=0.22,
    T_high=0.51,
    z_radius=1,
    xy_radius=0,
    dust_min_size=1279,
):
    """
    sample: (1, D, H, W, 1)
    """

    # # --------- ROTATION TTA PROBS ---------
    # probs_fg = predict_probs_tta_rot(sample,m)   # (D, H, W)
    probs_fg = sample
    if iid is not None:
        np.save(iid, probs_fg)
    # # --------- POSTPROCESS (UNCHANGED) ----
    final = topo_postprocess(
        probs_fg,
        T_low=T_low,
        T_high=T_high,
        z_radius=z_radius,
        xy_radius=xy_radius,
        dust_min_size=dust_min_size,
    )

    return final  # (D, H, W) uint8 {0,1}


In [3]:
# import os
# import glob
# import torch
# import tifffile
# import numpy as np
# import matplotlib.pyplot as plt
# import gc
# from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# # ================= SETUP PATHS =================
# DATASET_ROOT = "/kaggle/input/nnunet-v2"
# OUTPUT_DIR = "/kaggle/working"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Setup Environment Variables
# os.environ['nnUNet_raw'] = "/tmp/nnUNet_raw" 
# os.environ['nnUNet_preprocessed'] = "/tmp/nnUNet_preprocessed"
# os.environ['nnUNet_results'] = DATASET_ROOT
# os.makedirs(os.environ['nnUNet_raw'], exist_ok=True)

# # Auto-detect Trainer Folder
# trainer_dirs = glob.glob(os.path.join(DATASET_ROOT, "nnUNetTrainer*"))
# if not trainer_dirs:
#     raise ValueError("Could not find the nnUNetTrainer folder! Check DATASET_ROOT.")
    
# MODEL_FOLDER = trainer_dirs[0]
# print(f"✅ Found Model Folder: {os.path.basename(MODEL_FOLDER)}")

# # ================= INITIALIZE PREDICTOR =================
# print("⏳ Initializing nnU-Net Predictor...")
# predictor = nnUNetPredictor(
#     tile_step_size=0.5,
#     use_gaussian=True,
#     use_mirroring=True,
#     device=torch.device('cuda', 0),
#     verbose=False
# )

# predictor.initialize_from_trained_model_folder(
#     "/kaggle/input/nnunet-v2/nnUNetTrainer_SkelRecall__nnUNetResEncUNetMPlans__3d_fullres",
#     use_folds=(0,), 
#     checkpoint_name='checkpoint_best.pth',
# )
# print("🚀 Model Ready!")

In [4]:
import torch
import nnunetv2
import nnunetv2.utilities.find_class_by_name
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# ================= 1. DEFINE THE DUMMY CLASS =================
# This class mimics the one nnU-Net is desperately looking for.
class nnUNetTrainer_SkelRecall(nnUNetTrainer):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, unpack_dataset: bool = True, device: torch.device = torch.device('cuda')):
        # We just pass everything to the standard parent class
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)

# ================= 2. APPLY THE MONKEY PATCH =================
# We save the original search function so we don't break other things
original_recursive_find = nnunetv2.utilities.find_class_by_name.recursive_find_python_class

def patched_recursive_find(folder, trainer_name, current_module):
    # If nnU-Net asks for the missing class, give it our dummy one immediately
    if trainer_name == 'nnUNetTrainer_SkelRecall':
        print(f"🛡️ Intercepted search for '{trainer_name}'! returning dummy class.")
        return nnUNetTrainer_SkelRecall
    
    # Otherwise, behave normally
    return original_recursive_find(folder, trainer_name, current_module)

# Overwrite the library's function with our patched version
nnunetv2.utilities.find_class_by_name.recursive_find_python_class = patched_recursive_find
print("✅ Patch applied. nnU-Net's internal searcher has been tricked.")

# ================= 3. RUN PREDICTOR =================
# Now you can use the ORIGINAL read-only folder path. No copying needed.
MODEL_FOLDER = "/kaggle/input/nnunet-v2/nnUNetTrainer_SkelRecall__nnUNetResEncUNetMPlans__3d_fullres"

print("\n⏳ Initializing nnU-Net Predictor...")
predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    device=torch.device('cuda', 0),
    verbose=False
)

predictor.initialize_from_trained_model_folder(
    MODEL_FOLDER,
    use_folds=(0,),
    checkpoint_name='checkpoint_best.pth',
)

print("🚀 Model Ready! The error is bypassed.")

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.
✅ Patch applied. nnU-Net's internal searcher has been tricked.

⏳ Initializing nnU-Net Predictor...
🛡️ Intercepted search for 'nnUNetTrainer_SkelRecall'! returning dummy class.
🚀 Model Ready! The error is bypassed.


In [5]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def compare_3d_volume(
  vol1,
  vol2,
  axis=0,
  cmap="viridis"
):
  # vol1 = tifffile.imread(tiff_path_1)
  # vol2 = tifffile.imread(tiff_path_2)

  assert vol1.shape == vol2.shape
  assert axis in (0, 1, 2)

  vmin = min(vol1.min(), vol2.min())
  vmax = max(vol1.max(), vol2.max())

  def get_slice(vol, idx):
    if axis == 0:
      return vol[idx]
    elif axis == 1:
      return vol[:, idx, :]
    else:
      return vol[:, :, idx]

  slider = widgets.IntSlider(
    min=0,
    max=vol1.shape[axis] - 1,
    step=1,
    value=vol1.shape[axis] // 2,
    description="Slice",
    continuous_update=False
  )

  out = widgets.Output()

  def update(change=None):
    idx = slider.value
    with out:
      out.clear_output(wait=True)
      fig, axes = plt.subplots(1, 2, figsize=(12, 6))

      # REMOVE vmin=vmin, vmax=vmax to let matplotlib auto-scale
      im1 = axes[0].imshow(get_slice(vol1, idx), cmap=cmap)
      axes[0].set_title(f"Vol 1 (Auto-scaled) | Slice {idx}")
      plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
      axes[0].axis("off")

      im2 = axes[1].imshow(get_slice(vol2, idx), cmap=cmap)
      axes[1].set_title(f"Vol 2 (Auto-scaled) | Slice {idx}")
      plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
      axes[1].axis("off")

      plt.show()

      plt.show()

  slider.observe(update, names="value")

  display(widgets.VBox([slider, out]))
  update()


In [6]:
import torch
import numpy as np
import tifffile
import gc
def predict_output(list_tiff_files):
    # 1. Load the TIFF
    # Vesuvius .tifs are usually (Z, H, W). Ensure float32 for nnU-Net.
    base_path = "/kaggle/input/vesuvius-challenge-surface-detection/test_images/"
    img_list = [tifffile.imread(base_path + str(tif_path) + ".tif").astype(np.float32) for tif_path in list_tiff_files]
    img_list = [img.transpose(2, 1, 0) for img in img_list]
    img_list = [vol[np.newaxis,...] for vol in img_list]
    img = img_list[0]
    print(img.shape)
    # 2. Add the Channel Dimension
    # nnU-Net expects (Channels, Depth, Height, Width)
    # img[np.newaxis, ...] turns (Z, H, W) into (1, Z, H, W)
    if img.ndim == 3:
        img = img[np.newaxis, ...]
    props = {'spacing': [1.0, 1.0, 1.0]}
    list_props = [props]*len(list_tiff_files)

    print(img.shape)
    predictor_output = predictor.predict_from_list_of_npy_arrays(
        img_list,None,list_props,None,2,True,1
    )
    output_list = [predict(prob_map[1][1,:,:,:].transpose(2,1,0),T_low = 0.2,T_high = 0.8) for prob_map in predictor_output]
    del img_list,predictor_output
    # # 5. Post-processing
    # # Assuming 'predict' is your custom thresholding function
    # output = predict(prob_map, T_low=0.2, T_high=0.8)
    # print(output.shape)
    # # Cleanup to save Kaggle RAM
    # del img, logits, probabilities
    gc.collect()
    torch.cuda.empty_cache()
    
    return output_list


In [7]:
# %matplotlib inline
# compare_3d_volume(pred[0],tifffile.imread("/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"))

In [8]:
import os
import shutil
import tifffile
import numpy as np
import pandas as pd

# 1. Setup paths
root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
working_dir = "/kaggle/working"
# Temporary folder for .tif files
temp_sub_dir = os.path.join(working_dir, "temp_masks")
zip_path = os.path.join(working_dir, "submission.zip")

# Clean start
if os.path.exists(temp_sub_dir):
    shutil.rmtree(temp_sub_dir)
os.makedirs(temp_sub_dir, exist_ok=True)

test_df = pd.read_csv(f"{root_dir}/test.csv")
ids = test_df["id"].tolist()
batch_size = 25 

# 2. Prediction & Saving Loop
for i in range(0, len(ids), batch_size):
    id_list = ids[i : i + batch_size]
    print(f"🚀 Processing batch: {id_list}")
    
    output_list = predict_output(id_list)
    
    for j, mask in enumerate(output_list):
        current_id = id_list[j]
        # Save directly to the temporary folder
        out_path = os.path.join(temp_sub_dir, f"{current_id}.tif")
        
        # Ensure binary uint8 {0, 1}
        mask_binary = (mask).astype(np.uint8)
        tifffile.imwrite(out_path, mask_binary)

# 3. Create the ZIP (Only contains the .tif files)
print("📦 Archiving to submission.zip...")
# The -j flag junk-paths, but 'cd' ensures we are in the right spot
os.system(f"cd {temp_sub_dir} && zip -j {zip_path} ./*.tif")

# 4. EXTREME CLEANUP
# Remove the temporary masks folder
shutil.rmtree(temp_sub_dir, ignore_errors=True)

# Remove any other leftover directories (like the one you created earlier)
shutil.rmtree(os.path.join(working_dir, "submission_masks"), ignore_errors=True)
shutil.rmtree(os.path.join(working_dir, "_tta_preds"), ignore_errors=True)

# Remove any random .tif files that might have been saved in /working by mistake
for f in os.listdir(working_dir):
    full_path = os.path.join(working_dir, f)
    if f.endswith(".tif"):
        os.remove(full_path)

!rm -rf /kaggle/working/nnUNet_Patched_Model

print(f"✅ Finished! Current directory contents: {os.listdir(working_dir)}")

🚀 Processing batch: [1407735]
(1, 320, 320, 320)
(1, 320, 320, 320)
nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.

Predicting image of shape torch.Size([1, 314, 314, 320]):
perform_everything_on_device: True


100%|██████████| 27/27 [03:30<00:00,  7.80s/it]


sending off prediction to background worker for resampling

Done with image of shape torch.Size([1, 314, 314, 320]):
📦 Archiving to submission.zip...
  adding: 1407735.tif (deflated 98%)
✅ Finished! Current directory contents: ['__notebook__.ipynb', 'submission.zip']
